In [ ]:
from pathlib import Path
import importlib

BASE_DIR = Path.cwd().resolve()
MODULE_PATH = BASE_DIR.parent / "offline-detection" / "main.py"

offline_detection = importlib.machinery.SourceFileLoader(
    "offline-detection", str(MODULE_PATH)
).load_module()

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def msg_callback(msg, end=None):
	print(msg)
	time.sleep(1)

	if end is True: print("Done!")

def handle_report(news, **kwargs):
	preds = news["future_predictions"]

	bardata = np.unique(preds, return_counts=True)
	bar = plt.bar(bardata[0], bardata[1])

	plt.bar_label(bar)
	plt.xticks(ticks=[-1, 0, 1, 2], labels=[None, "Normal", "Anomaly", None])
	plt.title("Predictions of the pipeline")
	plt.show()

	metrics = pd.Series(news["metrics"])	

	core_metric_names = ["pr_auc", "precision", "recall", "f1"]
	core_metrics = metrics.reindex(core_metric_names).dropna()

	k_metric_names = ["k_ratio", "k", "precision_at_k", "recall_at_k"]
	k_metrics = metrics.reindex(k_metric_names).dropna()

	core_metric_labels = {
		"pr_auc": "PR AUC",
		"precision": "Precision",
		"recall": "Recall",
		"f1": "F1 Score",
	}
	core_metrics.rename(index=core_metric_labels, inplace=True)

	k_metric_labels = {
		"k_ratio": "Top-K Ratio",
		"k": "K (Top Items)",
		"precision_at_k": "Precision@K",
		"recall_at_k": "Recall@K",
	}
	k_metrics.rename(index=k_metric_labels, inplace=True)

	core_table = core_metrics.rename_axis("Core Metric").to_frame(name="Score")
	k_table = k_metrics.rename_axis("Top-K Metric").to_frame(name="Top-K Score")

	print("\nCore performance metrics:")
	print(core_table.T.to_markdown(floatfmt=".3f"))

	print("\nTop-K metrics:")
	if k_table.empty:
		print("No Top-K metrics available in this report.")
	else:
		print(k_table.T.to_markdown(floatfmt=".3f"))

In [ ]:
offline_detection.pipeline(
    # msg_callback=msg_callback,
    report_callback=handle_report,
    verb=False
);